In [14]:
%%time
%matplotlib inline
import rioxarray
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, make_scorer, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
import importlib 
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft, fftfreq
from scipy.interpolate import interp1d
from sklearn.linear_model import LinearRegression
import os
import glob

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq

CPU times: total: 0 ns
Wall time: 1.14 ms


In [2]:
# this code for calculating the bias and scale for each  land type
# Code tinh scale và bias

In [15]:
input_folder = "./input/local"
nonan_folder = "./input/non_nan"
output_folder = "./input/non_nan"

In [16]:
ndvi_files = [f for f in os.listdir(nonan_folder) if f.endswith('ndvi.npy')]
vh_files = [f for f in os.listdir(nonan_folder) if f.endswith('vh.npy')]

In [17]:
print(os.path.normpath(os.path.join(nonan_folder, ndvi_files[0])))

input\non_nan\ST_TS_nonan_ndvi.npy


In [21]:
def read_input_data(input_path):
    ndvi_array = np.load(input_path)
    mean_ndvi = ndvi_array.mean(axis=0)
    std_ndvi = ndvi_array.std(axis=0)
    return ndvi_array, mean_ndvi, std_ndvi


In [29]:
bias_range = np.arange(-5, 5, 0.01)

for i in range(len(vh_files)):
    ndvi_array,_, _ =  read_input_data(os.path.normpath(os.path.join(nonan_folder, ndvi_files[i])))
    vh_array, _, _ = read_input_data(os.path.normpath(os.path.join(nonan_folder, vh_files[i])))
    ndvi_interp_data = np.zeros_like(ndvi_array)
    scale_factors, interps = [], []
    bias = 1
    for x in range(vh_array.shape[0]):  # Lặp qua từng hàng (chuỗi thời gian)
        vh_fft = fft(vh_array[x])
        ndvi_fft = fft(ndvi_array[x])    
        # Tính hệ số tỷ lệ phổ
        scale_factor = np.abs(ndvi_fft) / (np.abs(vh_fft) + 1e-6)  # Tránh chia cho 0
        ndvi_interp_fft = vh_fft * scale_factor  # Áp dụng hệ số tỷ lệ
        scale_factors.append(scale_factor)
        interps.append(ndvi_interp_fft)
        # Nghịch biến đổi Fourier để khôi phục NDVI
        ndvi_interp_data[i] = np.real(ifft(ndvi_interp_fft)) + bias
        #print(scale_factor, ndvi_interp_fft)
    mean_scales = np.mean(scale_factors, axis=0)
    print("Best scale for " + ndvi_files[i])
    print(mean_scales)
    ##########################################################
    for y in range(vh_array.shape[0]):
        min_loss = float('inf')
        best_bias = 0
        scale_factor = np.abs(fft(ndvi_array[y])) / (np.abs(fft(vh_array[y])) + 1e-6)
        
        for bias in bias_range:
            ndvi_interp_fft = fft(vh_array[y]) * scale_factor
            ndvi_interp = np.real(ifft(ndvi_interp_fft)) + bias
            loss = np.mean((ndvi_interp - ndvi_array[y])**2)
            if loss < min_loss:
                min_loss = loss
                best_bias = bias
    print("Best bias")
    print(best_bias)
    print("#######################################")


Best scale for ST_TS_nonan_ndvi.npy
[0.00448667 0.09619107 0.04528891 0.08500575 0.06926585 0.08683764
 0.0714632  0.0714632  0.08683764 0.06926585 0.08500575 0.04528891
 0.09619107]
Best bias
1.3399999999998649
#######################################
Best scale for ST_CHN_nonan_ndvi.npy
[0.01195613 0.05535558 0.23118054 0.19157974 0.15663291 0.0683827
 0.13510587 0.13510587 0.0683827  0.15663291 0.19157974 0.23118054
 0.05535558]
Best bias
0.579999999999881
#######################################


2617